## Import libs and create controller

In [1]:
import json

from dotenv import load_dotenv

from src.app.controllers.app_controller import Controller
from src.prompts.system_prompt import system_prompt_gemini_answer_agent, system_prompt_gemini_query_preprocess_agent

In [2]:
load_dotenv()

controller = Controller(
    text_index_path="../data/faiss/articles_text.index",
    text_metadata_path="../data/metadata/metadata_article.pkl",
    image_index_path="../data/faiss/image.index",
    image_metadata_path="../data/metadata/metadata_image.pkl",
    qa_system_prompt=system_prompt_gemini_answer_agent,
    translator_system_prompt=system_prompt_gemini_query_preprocess_agent
)

## Load Data

In [9]:
def load_json_file(filepath: str):
    with open(filepath, 'r', encoding='utf-8') as f:
        return json.load(f)
    
json_results = load_json_file("rag_questions.json")

In [10]:
questions = []
ground_truth = []

for q in json_results:
    questions.append(q['question'])
    ground_truth.append(q['ground_truth'])

## Processing questions by Rag

In [ ]:
results = []

for idx, q in enumerate(questions, 1):
    print(f"Processing Q{idx}: {q}")
    try:
        urls_markdown, images, html_output, content = controller.retrieve(q, html_output_text=False, content=True)
        results.append({
            "question": q,
            "article_urls": urls_markdown,
            "image_urls": images,
            "html_answer": html_output,
            "content": content
        })
    except Exception as e:
        print(f"Error on question {idx}: {e}")
        results.append({
            "question": q,
            "error": str(e)
        })

In [21]:
def get_documents(input):
    contents = []
    for text in input:
        cleaned_content = BeautifulSoup(text['content'], "html.parser").get_text(separator="")
        contents.append(cleaned_content)
    
    return contents

In [22]:
from bs4 import BeautifulSoup

answers = []
documents = []

for answer in results:
    answers.append(answer['html_answer'])
    documents.append(get_documents(answer['content']))

## Create dataset for ragas evaluate

In [42]:
from datasets import Dataset

def prepare_ragas_dataset(qa, documents, reference_answers, model_answers):
    questions = []
    ground_truths = []
    answers = []
    contexts = []
    
    for entry, doc_list, ref, model_answer in zip(qa, documents, reference_answers, model_answers):
        questions.append(entry)
        ground_truths.append(ref)
        answers.append(model_answer)
    
        # Extract just the document text (ignore the scores)
        retrieved_contexts = [doc for doc in doc_list]
        contexts.append(retrieved_contexts)

    return Dataset.from_dict({
        "question": questions,
        "contexts": contexts,
        "answer": answers,
        "ground_truth": ground_truths
    })

In [43]:
res = prepare_ragas_dataset(qa=questions, documents=documents, reference_answers=ground_truth, model_answers=answers)

In [44]:
res

Dataset({
    features: ['question', 'contexts', 'answer', 'ground_truth'],
    num_rows: 15
})

## Evaluate old model 

In [50]:
from ragas import evaluate
from ragas.metrics import answer_relevancy, faithfulness, context_precision, context_recall

In [51]:
results = evaluate(
    res,
    metrics=[
        context_precision,
        context_recall,
        answer_relevancy,
        faithfulness,
    ]
)

print(results)

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:

{'context_precision': 0.6889, 'context_recall': 0.5833, 'answer_relevancy': 0.5414, 'faithfulness': 0.6889}
